In [60]:
using LowLevelFEM, LinearAlgebra

In [61]:
openGeometry("boxes.geo")

In [62]:
#openPreProcessor()

In [63]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);

In [64]:
bc_bottom = BoundaryCondition("bottom", ux=0, uy=0, uz=0)
bc_top = BoundaryCondition("top", ux=0, uz=0, uy=(x,y,z)->-x*(x-10) * z*(z-10) / 4250)

K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U))
f = ∫(U ⋅ [0, 0, 0])

@time u = solveField(K, f, support=[bc_bottom, bc_top])

showDoFResults(u, name="u", factor=1, visible=false)

  2.023640 seconds (116.05 k allocations: 465.491 MiB, 5.90% gc time, 4.95% compilation time)


0


## Penalty contact

The contact object contains only the contact geometry and kinematics. The full
contact operator maps the displacement field to the local contact space,

$$
G: V_u \rightarrow V_c ,
$$

while $P_a$ selects the currently active contact nodes,

$$
G_a=P_aG.
$$

The penalty surface operator is assembled once with the ordinary LLFEM weak-form
machinery and then restricted to the slave surface. In 2D the contact-space
ordering is $(n,t)$. For frictionless contact $c_t=0$.


In [65]:

r = nodePositionVector(U)

C = contact(
    u,
    master="master",
    slave="slave",
    topology_tol=0.01
)

cn = 1e7
ct = 0.0

Dc = [cn 0.0 0.0
      0.0 ct 0.0
      0.0 0.0 ct]

C0 = ∫(U ⋅ Dc ⋅ U, Γ="slave")
Cc = subSystemMatrix(C0; onPhysicalGroup="slave");



At a fixed contact geometry the active penalty contribution is

$$
d_a=P_a d,\qquad
C_a=P_a C_c P_a^T,\qquad
G_a=P_aG,
$$

$$
r_c=G_a^T C_a d_a,\qquad
K_c=G_a^T C_aG_a.
$$

Since $d=G(r+u)$, freezing the current geometry gives the Newton equation

$$
(K+K_c)u_{\mathrm{new}}=f-K_cr.
$$

Thus the linear correction can still be solved with the ordinary `solveField`
function. A line search is retained because the projection and active set may
change during the iteration.


In [66]:

support = [bc_bottom, bc_top]

support_increment = [
    BoundaryCondition("bottom", ux=0, uy=0, uz=0),
    BoundaryCondition("top",    ux=0, uy=0, uz=0)
]

free = freeDoFs(U, support)

# Only used to compare nonlinear residuals on unconstrained DoFs.
freeNorm(v::VectorField) = LinearAlgebra.norm(elementsToNodes(v).a[free, 1])

u_it = copy(u)

old_tags = copy(C.master_element_tags)
old_G = copy(C.G)

for iter in 1:30

    println("updateContact!")
    @time updateContact!(C, u_it)

    nchanged = count(old_tags .!= C.master_element_tags)

    dG = norm(C.G - old_G) /
         max(norm(old_G), eps())

    old_tags = copy(C.master_element_tags)
    old_G = copy(C.G)

    # Active contact algebra
    println("Ga")
    @time Ga = C.Pa * C.G
    println("Ca")
    @time Ca = C.Pa * Cc * C.Pa'
    println("da")
    @time da = C.Pa * C.d

    # Contact residual and frozen-geometry tangent
    println("rc")
    @time rc = Ga' * (Ca * da)
    println("Kc")
    @time Kc = Ga' * Ca * Ga

    # Total residual
    println("R")
    R = K * u_it - f + rc
    R0 = freeNorm(R)

    # Full frozen-geometry Newton step:
    # (K + Kc) u_new = f - Kc r
    println("solve")
    @time Δu = solveField(
        K + Kc,
        -R,
        support=support_increment
    )

    # Line search because G, projection and the active set change with u.
    α = 1.0

    @time u_trial = copy(u_it)
    Rtrial = R

    while α > 1e-6

        u_trial = u_it + α * Δu

        updateContact!(C, u_trial)

        Ga_trial = C.Pa * C.G
        Ca_trial = C.Pa * Cc * C.Pa'
        da_trial = C.Pa * C.d

        rc_trial = Ga_trial' * (Ca_trial * da_trial)

        Rtrial = K * u_trial - f + rc_trial

        freeNorm(Rtrial) < R0 && break

        α *= 0.5
    end

    u_it = u_trial

    err = freeNorm(α * Δu) /
          max(freeNorm(u_it), eps())

    println(
        "iter = ", iter,
        ", α = ", α,
        ", active = ", count(C.active),
        ", master changes = ", nchanged,
        ", dG = ", dG,
        ", min gap = ", minimum(C.gap_values),
        ", error = ", err,
        ", |R| = ", freeNorm(Rtrial)
    )

    err < 1e-8 && break
end

u = u_it

# Synchronize the stored contact state with the converged field.
updateContact!(C, u);


updateContact!
  0.437389 seconds (531.80 k allocations: 42.097 MiB)
Ga
  0.001017 seconds (33 allocations: 591.055 KiB)
Ca
  0.007342 seconds (85 allocations: 338.789 KiB, 94.11% gc time)
da
  0.000089 seconds (32 allocations: 13.867 KiB)
rc
  0.000314 seconds (59 allocations: 1.313 MiB)
Kc
  0.001785 seconds (50 allocations: 3.662 MiB)
R
solve
  2.601266 seconds (1.23 k allocations: 706.797 MiB, 4.22% gc time)
  0.000042 seconds (16 allocations: 384.820 KiB)
iter = 1, α = 1.0, active = 459, master changes = 0, dG = 3.5502757717324066e-14, min gap = -0.0002471697436751982, error = 0.059619907613795306, |R| = 468.1793469917796
updateContact!
  0.403224 seconds (531.67 k allocations: 43.545 MiB)
Ga
  0.000927 seconds (23 allocations: 688.180 KiB)
Ca
  0.000344 seconds (58 allocations: 394.977 KiB)
da
  0.000026 seconds (13 allocations: 11.383 KiB)
rc
  0.000243 seconds (41 allocations: 1.399 MiB)
Kc
  0.004495 seconds (50 allocations: 8.045 MiB)
R
solve
  3.086237 seconds (1.15 k alloca

In [67]:

showDoFResults(u, name="u", factor=1, visible=true)


1


## Contact fields

`C.d` is a reduced `ContactVector`. Mapping it back to the displacement mesh
makes the local contact components available through the ordinary field API.

For the current closest-point geometry, `D[1]` is the normal gap. The tangential
component of the current position difference is approximately zero by
construction; tangential slip will later be accumulated from displacement
increments/history.


In [68]:

Dn = VectorField(C.d)

gap = Dn[1]

# Active normal gap, expanded back to the full contact space.
da_full = C.Pa' * (C.Pa * C.d)
Da = VectorField(da_full)

# Pointwise penalty traction (positive in compression).
pressure = -cn * Da[1]

gap

showElementResults(nodesToElements(pressure, onPhysicalGroup="slave"), name="p")
showElementResults(nodesToElements(gap, onPhysicalGroup="slave"), name="gap")


3

In [69]:

# Example postprocessing:
# showElementResults(nodesToElements(gap), name="gap", visible=true)
# showElementResults(nodesToElements(pressure), name="pressure", visible=true)

openPostProcessor()
